# Task 3: Domain Generalization Launcher
Mount your Google Drive and navigate to the Task 3 directory.

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

# 1. Base Paths (Adjust DRIVE_ROOT if your new drive structure is different)
DRIVE_ROOT = '/content/drive/MyDrive/atml_assignment1'
TASK3_DIR = os.path.join(DRIVE_ROOT, 'Task 3')
ZIP_PATH = os.path.join(DRIVE_ROOT, 'data', 'PACS.zip') # Change this if you put the zip somewhere else!

# 2. Extract PACS to local Colab disk for FAST training (bypasses Google Drive bottleneck)
LOCAL_DATA_BASE = '/content/pacs_data'
if not os.path.exists(LOCAL_DATA_BASE):
    print("Extracting PACS.zip locally to speed up training...")
    !unzip -q "{ZIP_PATH}" -d "{LOCAL_DATA_BASE}"
else:
    print("Dataset already extracted locally.")

# 3. Auto-detect the exact PACS folder inside the local extraction
candidates = []
for root, dirs, files in os.walk(LOCAL_DATA_BASE):
    if set(['photo', 'art_painting', 'cartoon', 'sketch']).issubset(set(dirs)):
        candidates.append(root)
real = [c for c in candidates if 'dct' not in c.lower()]
DATA_DIR = real[0] if real else (candidates[0] if candidates else LOCAL_DATA_BASE)

# 4. Set up Task 2/3 directories
SPLITS_PATH = os.path.join(TASK3_DIR, 'shared', 'splits', 'pacs_sketch_seed6304.json')
TASK2_CKPT_DIR = os.path.join(DRIVE_ROOT, 'task2', 'checkpoints')
TASK3_CKPT_DIR = os.path.join(TASK3_DIR, 'checkpoints')
TASK3_RESULTS_DIR = os.path.join(TASK3_DIR, 'results')

os.makedirs(TASK3_CKPT_DIR, exist_ok=True)
os.makedirs(TASK3_RESULTS_DIR, exist_ok=True)

TASK2_CKPT_DIR = os.path.join(DRIVE_ROOT, 'task2', 'checkpoints')
ERM_CKPT = os.path.join(TASK2_CKPT_DIR, 'source_only_best.pth')
if not os.path.isfile(ERM_CKPT):
    print(f"⚠️  WARNING: ERM checkpoint not found at {ERM_CKPT}")
else:
    print(f"✓ ERM checkpoint found for warm-start!")
    
os.chdir(TASK3_DIR)
print(f'Working directory set to: {os.getcwd()}')
print(f'Data directory automatically set to: {DATA_DIR}')

Mounted at /content/drive
Extracting PACS.zip locally to speed up training...
✓ ERM checkpoint found for warm-start!
Working directory set to: /content/drive/MyDrive/atml_assignment1/Task 3
Data directory automatically set to: /content/pacs_data/pacs_data/pacs_data


In [9]:
import shutil
import os

print("Cleaning up old Task 3 checkpoints and results...")

# Delete the folders and everything inside them
if os.path.exists(TASK3_CKPT_DIR):
    shutil.rmtree(TASK3_CKPT_DIR)
    print(f"Deleted: {TASK3_CKPT_DIR}")

if os.path.exists(TASK3_RESULTS_DIR):
    shutil.rmtree(TASK3_RESULTS_DIR)
    print(f"Deleted: {TASK3_RESULTS_DIR}")

# Recreate the empty folders so the training script doesn't complain
os.makedirs(TASK3_CKPT_DIR, exist_ok=True)
os.makedirs(TASK3_RESULTS_DIR, exist_ok=True)

print("Clean up complete! You are ready for a fresh run.")

Cleaning up old Task 3 checkpoints and results...
Deleted: /content/drive/MyDrive/atml_assignment1/Task 3/checkpoints
Deleted: /content/drive/MyDrive/atml_assignment1/Task 3/results
Clean up complete! You are ready for a fresh run.


### Step 1: Train the Core Models (DAN-DG & SAM)

In [10]:
!python train.py \
  --method dan_dg \
  --data_root "{DATA_DIR}" \
  --splits_path "{SPLITS_PATH}" \
  --checkpoints_dir "{TASK3_CKPT_DIR}" \
  --results_dir "{TASK3_RESULTS_DIR}" \
  --erm_ckpt "{ERM_CKPT}"


[train] Method: dan_dg  |  Device: cuda
[train] Run name: dan_dg
[pacs] Dataset found at /content/pacs_data/pacs_data/pacs_data
[protocol] Creating stratified 80/20 splits (seed 6304)…
  [photo] total=1670  train=1336  val=334
  [art_painting] total=2048  train=1638  val=410
  [cartoon] total=2344  train=1875  val=469
[protocol] Splits saved → /content/drive/MyDrive/atml_assignment1/Task 3/shared/splits/pacs_sketch_seed6304.json
[train] Warm-started backbone+head from ERM: /content/drive/MyDrive/atml_assignment1/task2/checkpoints/source_only_best.pth
[train] Steps/epoch: 167  |  Total steps: 5010
Epoch [  1/30]  cls_loss=0.1423  mmd_loss=0.5005  total_loss=0.2424  | mean_src_F1=0.9024  ← best
  [ckpt] Saved → /content/drive/MyDrive/atml_assignment1/Task 3/checkpoints/dan_dg_best.pth  (42.7 MB)
Epoch [  2/30]  cls_loss=0.1611  mmd_loss=0.5113  total_loss=0.3656  | mean_src_F1=0.9039  ← best
  [ckpt] Saved → /content/drive/MyDrive/atml_assignment1/Task 3/checkpoints/dan_dg_best.pth  (42.

In [2]:
!python train.py \
  --method sam \
  --data_root "{DATA_DIR}" \
  --splits_path "{SPLITS_PATH}" \
  --checkpoints_dir "{TASK3_CKPT_DIR}" \
  --results_dir "{TASK3_RESULTS_DIR}" \
  --erm_ckpt "{ERM_CKPT}"


[train] Method: sam  |  Device: cuda
[train] Run name: sam
[pacs] Dataset found at /content/pacs_data/pacs_data/pacs_data
[protocol] Loading splits from /content/drive/MyDrive/atml_assignment1/Task 3/shared/splits/pacs_sketch_seed6304.json
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 219MB/s]
[train] Warm-started backbone+head from ERM: /content/drive/MyDrive/atml_assignment1/task2/checkpoints/source_only_best.pth
[train] Steps/epoch: 167  |  Total steps: 5010
Epoch [  1/30]  cls_loss=0.3512  | mean_src_F1=0.9318  ← best
  [ckpt] Saved → /content/drive/MyDrive/atml_assignment1/Task 3/checkpoints/sam_best.pth  (42.7 MB)
Epoch [  2/30]  cls_loss=0.2420  | mean_src_F1=0.9200
Epoch [  3/30]  cls_loss=0.2428  | mean_src_F1=0.9410  ← best
  [ckpt] Saved → /content/drive/MyDrive/atml_assignment1/Task 3/checkpoints/sam_best.pth  (42.7 MB)
Epoch [  4/30]  cls_loss=0.2024  | mea

### Step 2: Controlled Study (Lambda Sweep for DAN-DG)

In [2]:
# Lambda = 0.1
!python train.py --method dan_dg --lambda_dg 0.1 \
  --data_root "{DATA_DIR}" \
  --splits_path "{SPLITS_PATH}" \
  --checkpoints_dir "{TASK3_CKPT_DIR}" \
  --results_dir "{TASK3_RESULTS_DIR}" \
  --erm_ckpt "{ERM_CKPT}"


[train] Method: dan_dg  |  Device: cuda
[train] Run name: dan_dg_lambda_dg0.1
[pacs] Dataset found at /content/pacs_data/pacs_data/pacs_data
[protocol] Loading splits from /content/drive/MyDrive/atml_assignment1/Task 3/shared/splits/pacs_sketch_seed6304.json
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 102MB/s] 
[train] Warm-started backbone+head from ERM: /content/drive/MyDrive/atml_assignment1/task2/checkpoints/source_only_best.pth
[train] Steps/epoch: 167  |  Total steps: 5010
Epoch [  1/30]  cls_loss=0.1319  mmd_loss=0.5064  total_loss=0.1421  | mean_src_F1=0.9112  ← best
  [ckpt] Saved → /content/drive/MyDrive/atml_assignment1/Task 3/checkpoints/dan_dg_lambda_dg0.1_best.pth  (42.7 MB)
Epoch [  2/30]  cls_loss=0.1195  mmd_loss=0.4995  total_loss=0.1395  | mean_src_F1=0.9107
Epoch [  3/30]  cls_loss=0.0769  mmd_loss=0.4735  total_loss=0.1053  | mean_src_F1=0.9355  ←

In [3]:
# Lambda = 1.0 (Baseline)
!python train.py --method dan_dg --lambda_dg 1.0 \
  --data_root "{DATA_DIR}" \
  --splits_path "{SPLITS_PATH}" \
  --checkpoints_dir "{TASK3_CKPT_DIR}" \
  --results_dir "{TASK3_RESULTS_DIR}" \
  --erm_ckpt "{ERM_CKPT}"


[train] Method: dan_dg  |  Device: cuda
[train] Run name: dan_dg_lambda_dg1.0
[pacs] Dataset found at /content/pacs_data/pacs_data/pacs_data
[protocol] Loading splits from /content/drive/MyDrive/atml_assignment1/Task 3/shared/splits/pacs_sketch_seed6304.json
[train] Warm-started backbone+head from ERM: /content/drive/MyDrive/atml_assignment1/task2/checkpoints/source_only_best.pth
[train] Steps/epoch: 167  |  Total steps: 5010
Epoch [  1/30]  cls_loss=0.1423  mmd_loss=0.5005  total_loss=0.2424  | mean_src_F1=0.9024  ← best
  [ckpt] Saved → /content/drive/MyDrive/atml_assignment1/Task 3/checkpoints/dan_dg_lambda_dg1.0_best.pth  (42.7 MB)
Epoch [  2/30]  cls_loss=0.1611  mmd_loss=0.5113  total_loss=0.3656  | mean_src_F1=0.9039  ← best
  [ckpt] Saved → /content/drive/MyDrive/atml_assignment1/Task 3/checkpoints/dan_dg_lambda_dg1.0_best.pth  (42.7 MB)
Epoch [  3/30]  cls_loss=0.1868  mmd_loss=0.4997  total_loss=0.4866  | mean_src_F1=0.9155  ← best
  [ckpt] Saved → /content/drive/MyDrive/atml

In [4]:
# Lambda = 10.0
!python train.py --method dan_dg --lambda_dg 10.0 \
  --data_root "{DATA_DIR}" \
  --splits_path "{SPLITS_PATH}" \
  --checkpoints_dir "{TASK3_CKPT_DIR}" \
  --results_dir "{TASK3_RESULTS_DIR}" \
  --erm_ckpt "{ERM_CKPT}"


[train] Method: dan_dg  |  Device: cuda
[train] Run name: dan_dg_lambda_dg10.0
[pacs] Dataset found at /content/pacs_data/pacs_data/pacs_data
[protocol] Loading splits from /content/drive/MyDrive/atml_assignment1/Task 3/shared/splits/pacs_sketch_seed6304.json
[train] Warm-started backbone+head from ERM: /content/drive/MyDrive/atml_assignment1/task2/checkpoints/source_only_best.pth
[train] Steps/epoch: 167  |  Total steps: 5010
Epoch [  1/30]  cls_loss=0.3177  mmd_loss=0.5065  total_loss=1.3308  | mean_src_F1=0.7959  ← best
  [ckpt] Saved → /content/drive/MyDrive/atml_assignment1/Task 3/checkpoints/dan_dg_lambda_dg10.0_best.pth  (42.7 MB)
Epoch [  2/30]  cls_loss=0.9213  mmd_loss=0.5039  total_loss=2.9370  | mean_src_F1=0.7303
Epoch [  3/30]  cls_loss=1.8757  mmd_loss=0.4971  total_loss=4.8583  | mean_src_F1=0.4997
Epoch [  4/30]  cls_loss=3.6253  mmd_loss=0.4807  total_loss=7.4708  | mean_src_F1=0.0372
Epoch [  5/30]  cls_loss=4.6513  mmd_loss=0.2865  total_loss=7.5163  | mean_src_F1=0

### Step 3: Final Sketch Evaluation

In [5]:
!python evaluate_sketch.py \
  --data_root "{DATA_DIR}" \
  --splits_path "{SPLITS_PATH}" \
  --checkpoints_dir "{TASK3_CKPT_DIR}" \
  --task2_checkpoints_dir "{TASK2_CKPT_DIR}" \
  --results_dir "{TASK3_RESULTS_DIR}"


[eval] Using device: cuda
[pacs] Dataset found at /content/pacs_data/pacs_data/pacs_data
[protocol] Loading splits from /content/drive/MyDrive/atml_assignment1/Task 3/shared/splits/pacs_sketch_seed6304.json
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

[eval] Evaluating erm...
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that